# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Accessing metadata attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}; Version: {dataset.metadata.version}")
print(f"Keywords: {', '.join(dataset.metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

This section uses record set and field `@id`s for reference, ensuring reproducibility and clarity.

In [ ]:
# List all record sets in the dataset
record_sets = dataset.metadata.record_sets
print("Available record sets and their @id values:")
for record_set in record_sets:
    print(f"- name: {record_set.name}, @id: {record_set.id}")

# Review fields and columns for each record set
for record_set in record_sets:
    print(f"\nRecord set: {record_set.name} (@id: {record_set.id})")
    if hasattr(record_set, 'fields') and record_set.fields:
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - {field.name} (@id: {field.id})  [dataType: {getattr(field, 'data_type', None)}]")
    if hasattr(record_set, 'columns') and record_set.columns:
        print("  Columns:")
        for column in record_set.columns:
            print(f"    - {column.name} (@id: {column.id})  [dataType: {getattr(column, 'data_type', None)}]")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract the main data record set and display its structure.

In [ ]:
# Put the @id strings for the record sets you want to extract
record_set_ids = [recset.id for recset in dataset.metadata.record_sets]
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"\nFirst 5 rows of DataFrame for record set '@id': {rsid}")
    if not df.empty:
        print(df.head())
    else:
        print("(No data loaded)")

# Display available columns of the main record set (replace with correct @id if needed):
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print("\nColumns in main record set DataFrame (@id: {}):".format(main_record_set_id))
    print(dataframes[main_record_set_id].columns.tolist())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, and grouping data by attributes to prepare it for further analysis.

Below, we select a numeric field by its `@id` and demonstrate filtering, normalizing, and grouping (if applicable).

In [ ]:
# Example: select numeric field and group field by their @id (edit as appropriate from your overview above)
main_record_set = dataset.metadata.record_sets[0] if dataset.metadata.record_sets else None
if main_record_set and hasattr(main_record_set, 'fields'):
    numeric_field = None
    group_field = None
    for field in main_record_set.fields:
        # Heuristically pick the first numeric field (e.g. MW or coverage or abundance, with type Float/Integer/Number)
        if getattr(field, 'data_type', None) in ('Float', 'Integer', 'Number') and numeric_field is None:
            numeric_field = field.id
        # Heuristically pick first string field for grouping (e.g. description/class)
        if getattr(field, 'data_type', None) in ('Text', 'String') and group_field is None:
            group_field = field.id

    df = dataframes[main_record_set.id]
    if numeric_field in df.columns:
        # Try to use numeric type for the column
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

        threshold = df[numeric_field].mean() # Use mean as threshold for demonstration
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where '{numeric_field}' > {threshold} (mean):")
        print(filtered_df.head())

        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, normalized_col]].head())

        # Group by group_field if available
        if group_field and group_field in df.columns:
            group_cols = [col for col in [numeric_field, normalized_col] if col in filtered_df.columns]
            grouped_df = filtered_df.groupby(group_field)[group_cols].mean()
            print(f"\nGrouped data by '{group_field}':")
            print(grouped_df.head())
    else:
        print("No numeric field detected in the DataFrame for EDA.")
else:
    print("Main record set or fields not found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Example: Plot distribution of the numeric field (e.g., protein abundance or MW), and potentially relationships (e.g., grouped mean).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field if available
if main_record_set and hasattr(main_record_set, 'fields') and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field and group_field in df.columns:
        # Boxplot of numeric field by group
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"'{numeric_field}' by '{group_field}'")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field to visualize.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the dataset using the Croissant schema and `mlcroissant`.
- The structure and field mapping were reviewed via their `@id` values, ensuring reproducible references.
- Numeric field analysis and basic visualizations highlighted the record set's main characteristics, which can be further tailored as needed based on the research question.